## 1. Environment Setup & Imports

In [2]:
import sys
import os
import json
from pathlib import Path
from typing import List, Dict
import pandas as pd
# Ensure project root is on sys.path
project_root = Path.cwd()
sys.path.insert(0, str(project_root))
# Import project modules
from src.ingest import extract_pages_from_pdf, chunk_text, ingest_pdfs
from src.indexer import load_chunks, build_faiss_index
from src.query import Retriever, synthesize_answer, rerank_results
from src.evaluate import load_eval_set, precision_at_k
print("✓ All imports successful")
print(f"Project root: {project_root}")

C:\Users\moham\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All imports successful
Project root: c:\Users\moham\Downloads\OneDrive_2026-05-16\Document Navigator - A Transparent RAG Assistant


## 2. Check Directory Structure

In [3]:
pdf_dir = project_root / "pdfs"
data_dir = project_root / "data"
chunks_path = data_dir / "chunks.jsonl"
index_path = data_dir / "faiss.index"
eval_csv = project_root / "eval_set.csv"
print(f"PDF directory exists: {pdf_dir.exists()}")
print(f"Data directory exists: {data_dir.exists()}")
print(f"Chunks file exists: {chunks_path.exists()}")
print(f"Index exists: {index_path.exists()}")
print(f"Eval set exists: {eval_csv.exists()}")
# List PDFs
pdf_files = list(pdf_dir.glob("*.pdf"))
print(f"\nFound {len(pdf_files)} PDFs:")
for pdf in pdf_files:
    print(f" - {pdf.name}")

PDF directory exists: True
Data directory exists: True
Chunks file exists: True
Index exists: True
Eval set exists: True

Found 12 PDFs:
 - guide_chunking_strategy.pdf
 - guide_evaluation_metrics.pdf
 - guide_logging_monitoring.pdf
 - guide_rag_basics.pdf
 - guide_support_escalation.pdf
 - guide_system_prompting.pdf
 - guide_vector_search.pdf
 - MohammedwasiullahResume (2) (1) (1) (1) (1).pdf
 - Payslip_Apr_2026.pdf
 - policy_payments_security.pdf
 - policy_privacy_data_use.pdf
 - policy_shipping_returns.pdf


## 3. Ingestion & Chunking Pipeline

In [4]:
# If chunks don't exist, ingest PDFs
if not chunks_path.exists():
    print("Ingesting PDFs...")
    ingest_pdfs(str(pdf_dir), str(chunks_path), chunk_size=1000, overlap=200)
    print("✓ Ingestion complete")
else:
    print(f"Chunks already exist at {chunks_path}")
# Load and display chunks
chunks = load_chunks(str(chunks_path))
print(f"\nLoaded {len(chunks)} chunks")
print("\nSample chunk (first):")
print(json.dumps(chunks[0], indent=2, ensure_ascii=False)[:500] + "...")

Chunks already exist at c:\Users\moham\Downloads\OneDrive_2026-05-16\Document Navigator - A Transparent RAG Assistant\data\chunks.jsonl

Loaded 17 chunks

Sample chunk (first):
{
  "chunk_id": "C000001",
  "source": "MohammedwasiullahResume (2) (1) (1) (1) (1).pdf",
  "page": 1,
  "chunk_index": 0,
  "text": "Education  \n  \n  \n \n \n \n  \n  \n  \n \n \n \n  \n \n \n• \n \n• \n \n• \nProjects  \n   \n•  \n• \n \n• \n• \n  \n• \n \n \n• \n \n• \n        \n• \n• \n• \n  \n \n  \n  \nCertifications   \n•   \n•                                                                                                             \n• \n• \n \n \n \nMohammed Wasiullah\nPH: 9606468360...


## 4. Build or Load FAISS Index

In [5]:
if not index_path.exists():
    print("Building FAISS index...")
    build_faiss_index(str(chunks_path), str(index_path))
    print("Index built and saved")
else:
    print(f"Index already exists at {index_path}")
# Load the retriever
retriever = Retriever(str(index_path))
print("> Retriever loaded, ready for queries")

Index already exists at c:\Users\moham\Downloads\OneDrive_2026-05-16\Document Navigator - A Transparent RAG Assistant\data\faiss.index


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 353.91it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


> Retriever loaded, ready for queries


## 5. Sample Query with Retrieval Traces

In [6]:
test_query = "What is the standard delivery timeline?"
print(f"Query: {test_query}\n")
ret = retriever.query(test_query, top_k=5)
print("Retrieval Traces (Top 5):")
print("-" * 100)
for i, r in enumerate(ret["results"], 1):
    print(f"\n{i}. Chunk {r['chunk_id']} | Source: {r['source']}:{r['page']} | Score: {r['score']:.4f}")
    print(f"Text: {r['text'][:150]}...")

Query: What is the standard delivery timeline?

Retrieval Traces (Top 5):
----------------------------------------------------------------------------------------------------

1. Chunk C000017 | Source: policy_shipping_returns.pdf:1 | Score: 0.4201
Text: Shipping & Returns Policy (Sample)
1. Standard shipping is free for orders above INR 999; otherwise a flat fee of INR 79 applies.
2. Standard delivery...

2. Chunk C000015 | Source: policy_payments_security.pdf:1 | Score: 0.2866
Text: Payments & Security Policy (Sample)
1. Accepted payment methods include cards, UPI, net banking, and select wallets.
2. Cash on Delivery may be availa...

3. Chunk C000008 | Source: guide_chunking_strategy.pdf:1 | Score: 0.1799
Text: Chunking Strategy Notes (Sample)
1. Typical chunk sizes for narrative PDFs range from 500 to 800 tokens.
2. Overlaps of 50 to 100 tokens help preserve...

4. Chunk C000016 | Source: policy_privacy_data_use.pdf:1 | Score: 0.1474
Text: Privacy & Data Use Policy (Sample)
1. Coll

## 6. Answer Synthesis with Citations

In [7]:
synth = synthesize_answer(ret, score_threshold=0.25)
print("ANSWER:")
print("=" * 100)
print(synth["answer"])
print("=" * 100)
print(f"\nLow confidence: {synth['low_confidence']}")
print(f"Citations: {synth['citations']}")

ANSWER:
- Shipping & Returns Policy (Sample)
- 1. Standard shipping is free for orders above INR 999; otherwise a flat fee of INR 79 applies.
- 2. Standard delivery takes 3–6 business days depending on location.
- 3. Most products can be returned within 7 days if unused and in original packaging.
- 4. Refunds are processed within 3–7 business days after quality check.
- This PDF is synthetic and intended for capstone evaluation only.

Citations: [policy_shipping_returns.pdf:1]; [policy_payments_security.pdf:1]; [guide_chunking_strategy.pdf:1]; [policy_privacy_data_use.pdf:1]; [MohammedwasiullahResume (2) (1) (1) (1) (1).pdf:1]

Low confidence: False
Citations: ['[policy_shipping_returns.pdf:1]', '[policy_payments_security.pdf:1]', '[guide_chunking_strategy.pdf:1]', '[policy_privacy_data_use.pdf:1]', '[MohammedwasiullahResume (2) (1) (1) (1) (1).pdf:1]']


## 7. Filtered & Deduplicated Retrieval

In [ ]:
# 7. Filtered & Deduplicated Retrieval
threshold = 0.25
ret_filtered = retriever.query(test_query, top_k=5)
# Filter by threshold
ret_filtered["results"] = [r for r in ret_filtered["results"] if r["score"] >= threshold]
# Deduplicate by source/page
seen = set()
filtered = []
for r in ret_filtered["results"]:
    key = (r["source"], r["page"])
    if key in seen:
        continue
    seen.add(key)
    filtered.append(r)
ret_filtered["results"] = filtered
print(f"After filtering (threshold >= {threshold}) and deduplication:")
print(f"Results: {len(ret_filtered['results'])} unique chunks")
for i, r in enumerate(ret_filtered["results"], 1):
    print(f"{i}. {r['source']}:{r['page']} (score {r['score']:.4f})")

After filtering (threshold >= 0.25) and deduplication:
Results: 2 unique chunks
  1. policy_shipping_returns.pdf:1 (score 0.4201)
  2. policy_payments_security.pdf:1 (score 0.2866)


## 8. Optional Reranking (Cross-Encoder)

In [10]:
# Optional: Reranking with Cross-Encoder (skip by default to avoid memory issues)
# If you have sufficient RAM (8GB+), set skip_reranking=False

skip_reranking = True  # Set to False only if you have 8GB+ RAM available

if skip_reranking:
    print("⊘ Reranking skipped (set skip_reranking=False if you have 8GB+ RAM)")
else:
    try:
        print("Attempting to rerank with cross-encoder...")
        ret_reranked = retriever.query(test_query, top_k=10)
        ret_reranked["results"] = [r for r in ret_reranked["results"] if r["score"] >= threshold]
        ret_reranked["results"] = rerank_results(test_query, ret_reranked["results"])
        
        print("\nAfter reranking (top 5):")
        for i, r in enumerate(ret_reranked["results"][:5], 1):
            rerank_score = r.get("rerank_score", "N/A")
            print(f"  {i}. {r['source']}:{r['page']} | Original score: {r['score']:.4f} | Rerank score: {rerank_score}")
    except Exception as e:
        print(f"⚠ Reranking failed (insufficient memory or model download issue): {e}")
        print("  Hint: This feature requires 8GB+ RAM. Skipping reranking.")

⊘ Reranking skipped (set skip_reranking=False if you have 8GB+ RAM)


## 9. Evaluation Set & Precision Metrics

In [1]:
print("Loading evaluation CSV...")
try:
    eval_df = load_eval_set(str(eval_csv))
    print(f"✓ Evaluation set loaded: {len(eval_df)} questions")
    required_cols = ["id", "question", "gold_citation", "gold_key_phrase"]
    missing = [c for c in required_cols if c not in eval_df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    print(f"✓ All required columns present: {eval_df.columns.tolist()}")
    print("\nFirst 5 questions with gold citations:")
    print(eval_df[["id", "question", "gold_citation", "gold_key_phrase"]].head())
except Exception as e:
    print(f"✗ Failed to load evaluation set: {e}")
    raise

Loading evaluation CSV...
✗ Failed to load evaluation set: name 'load_eval_set' is not defined


NameError: name 'load_eval_set' is not defined

## 10. Compute Precision@k

In [11]:
from src.evaluate import precision_at_k

# Compute metrics
p3 = precision_at_k(str(eval_csv), str(index_path), k=3)
p5 = precision_at_k(str(eval_csv), str(index_path), k=5)

print("Retrieval Evaluation Results")
print("=" * 50)
print(f"Precision@3: {p3:.3f}")
print(f"Precision@5: {p5:.3f}")
print("\nInterpretation:")
print(f"  - {p3*100:.1f}% of questions have correct citation in top-3 results")
print(f"  - {p5*100:.1f}% of questions have correct citation in top-5 results")

: 

## 11. Example Success Case

In [ ]:
# Run a success example
success_q = eval_df.iloc[0]
print(f"Question: {success_q['question']}")
print(f"Expected (gold): {success_q['gold_citation']}")
print(f"Key phrase: {success_q['gold_key_phrase']}\n")

# Retrieve
ret_ex = retriever.query(success_q['question'], top_k=5)
print("Retrieved (top 3):")
for i, r in enumerate(ret_ex["results"][:3], 1):
    citation = f"{r['source']}:{r['page']}"
    match = "✓ MATCH" if citation == success_q['gold_citation'] else "✗ NO MATCH"
    print(f"  {i}. {citation} (score {r['score']:.4f}) {match}")
    print(f"     Text: {r['text'][:80]}...\n")

## 12. Answer Quality & Low-Confidence Handling

In [ ]:
# Compare different threshold settings
test_q = "What is a system prompt used for?"
ret_test = retriever.query(test_q, top_k=5)

print(f"Query: {test_q}")
print(f"\nTop 3 scores:")
for r in ret_test["results"][:3]:
    print(f"  {r['source']}:{r['page']} -> score {r['score']:.4f}")

# Synthesize with low threshold
synth_low = synthesize_answer(ret_test, score_threshold=0.1)
print(f"\nWith threshold 0.1:")
print(f"  Low confidence: {synth_low['low_confidence']}")
print(f"  Answer: {synth_low['answer'][:100]}...")

# Synthesize with high threshold
synth_high = synthesize_answer(ret_test, score_threshold=0.5)
print(f"\nWith threshold 0.5:")
print(f"  Low confidence: {synth_high['low_confidence']}")
print(f"  Answer: {synth_high['answer'][:100]}...")

## 13. Batch Evaluation on All Questions

In [ ]:
# Run all evaluation questions
results_list = []
for idx, row in eval_df.iterrows():
    q = row['question']
    gold = row['gold_citation']
    
    ret = retriever.query(q, top_k=5)
    retrieved_citations = [f"{r['source']}:{r['page']}" for r in ret["results"]]
    
    found_in_3 = gold in retrieved_citations[:3]
    found_in_5 = gold in retrieved_citations[:5]
    
    results_list.append({
        "id": row['id'],
        "question": q[:50],
        "gold": gold,
        "top_3_match": found_in_3,
        "top_5_match": found_in_5,
        "top_retrieved": retrieved_citations[0] if retrieved_citations else "None"
    })

results_df = pd.DataFrame(results_list)
print("Batch Evaluation Results:")
print(results_df.to_string())
print(f"\nSummary:")
print(f"  Top-3 matches: {results_df['top_3_match'].sum()}/{len(results_df)}")
print(f"  Top-5 matches: {results_df['top_5_match'].sum()}/{len(results_df)}")

## 14. Summary & Key Insights

In [ ]:
print("Document Navigator — Summary")
print("=" * 80)
print(f"\n✓ Ingestion: {len(chunks)} chunks from {len(pdf_files)} PDFs")
print(f"✓ Indexing: FAISS index with sentence-transformers embeddings")
print(f"✓ Retrieval: Semantic search + optional reranking")
print(f"✓ Citations: [filename:page] format")
print(f"✓ Evaluation: Precision@3={p3:.3f}, Precision@5={p5:.3f}")
print(f"\n✓ Features:")
print(f"  - Transparent retrieval traces (chunk_id, source, page, score)")
print(f"  - Low-confidence handling with configurable thresholds")
print(f"  - Deduplication by source/page")
print(f"  - Optional cross-encoder reranker")
print(f"  - Batch evaluation metrics")
print(f"\n✓ Output Formats:")
print(f"  - eval_set.csv: labeled questions with gold citations")
print(f"  - retrieval_report.md: evaluation results and recommendations")
print(f"  - Streamlit app: interactive Q&A with UI controls")
print(f"  - Notebook (this): end-to-end pipeline walkthrough")